In [1]:
from pathlib import Path
import pandas as pd

def find_project_root():
    path = Path.cwd().resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError("Could not find data/raw from " + str(path))

RAW_DIR = find_project_root() / "data" / "raw"

def read_xpt(period, name):
    return pd.read_sas(RAW_DIR / period / f"{name}.xpt", format="xport").round(6)

# Expected variables per component (D-006)
EXPECTED = {
    "DEMO":   ["RIAGENDR", "RIDAGEYR"],
    "BMX":    ["BMXBMI", "BMXWAIST"],
    "SMQ":    ["SMQ020", "SMQ040"],
    "PAQ":    ["PAQ605", "PAQ620", "PAQ635", "PAQ650", "PAQ665"],
    "DIQ":    ["DIQ010"],
    "MCQ":    ["MCQ300A", "MCQ300C"],
    "GHB":    ["LBXGH"],
    "TCHOL":  ["LBXTC"],
    "HDL":    ["LBDHDD"],
    "BIOPRO": ["LBXSCR"],
}
FILE_NAME = {
    "2017-2020": lambda c: f"P_{c}",
    "2021-2023": lambda c: f"{c}_L",
}

rows = []
for comp, cols in EXPECTED.items():
    for period, fname in FILE_NAME.items():
        df = read_xpt(period, fname(comp))
        upper = {c.upper(): c for c in df.columns}
        for col in cols:
            rows.append({"component": comp, "variable": col, "period": period,
                         "found": col in upper})

check = pd.DataFrame(rows).pivot_table(index=["component", "variable"],
                                       columns="period", values="found", aggfunc="first")
print(check)

# For any component with a missing variable, list what the file actually contains
for comp, cols in EXPECTED.items():
    for period, fname in FILE_NAME.items():
        df = read_xpt(period, fname(comp))
        upper = {c.upper() for c in df.columns}
        if not all(c in upper for c in cols):
            print(f"\n{fname(comp)} columns:", sorted(df.columns))

period              2017-2020  2021-2023
component variable                      
BIOPRO    LBXSCR         True       True
BMX       BMXBMI         True       True
          BMXWAIST       True       True
DEMO      RIAGENDR       True       True
          RIDAGEYR       True       True
DIQ       DIQ010         True       True
GHB       LBXGH          True       True
HDL       LBDHDD         True       True
MCQ       MCQ300A        True      False
          MCQ300C        True      False
PAQ       PAQ605         True      False
          PAQ620         True      False
          PAQ635         True      False
          PAQ650         True      False
          PAQ665         True      False
SMQ       SMQ020         True       True
          SMQ040         True       True
TCHOL     LBXTC          True       True

PAQ_L columns: ['PAD680', 'PAD790Q', 'PAD790U', 'PAD800', 'PAD810Q', 'PAD810U', 'PAD820', 'SEQN']

MCQ_L columns: ['AGQ030', 'MCQ010', 'MCQ035', 'MCQ040', 'MCQ050', 'MCQ053', 'MCQ